### 环境配置
MindSpore 2.3 
MindNLP 0.3.1
Python 3.9

In [ ]:
%%capture captured_output
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

安装完成后重启kernel，选择python3.9

In [ ]:
%%capture captured_output
!pip uninstall mindspore-gpu -y
!pip install https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.3/MindSpore/unified/x86_64/mindspore-2.3-cp39-cp39-linux_x86_64.whl --trusted-host ms-release.obs.cn-north-4.myhuaweicloud.com -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install download nltk -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install mindnlp
!pip install matplotlib

### 使用 Perceiver 处理文本
在这里，我们加载 PerceiverForMaskedLM 模型，它的训练方式与 BertForMaskedLM 类似。不过，PerceiverForMaskedLM 可以基于原始的 UTF - 8 字节进行训练，而非像 BERT 那样使用子词（BERT 采用 WordPiece 子词分词法）。模型的最大序列长度被设定为 2048 字节，这是论文作者为了能与 BERT 的 512 个子词进行公平对比而如此设置的。

参考论文：https://arxiv.org/abs/2107.14795

我们首先使用 PerceiverTokenizer 为模型准备文本。

In [ ]:
from mindnlp.transformers.models import PerceiverTokenizer, PerceiverForMaskedLM

tokenizer = PerceiverTokenizer.from_pretrained("deepmind/language-perceiver")
model = PerceiverForMaskedLM.from_pretrained("deepmind/language-perceiver")

在这里，我们对文本进行 “分词” 操作（这实际上会将文本转换为一个字节 ID 序列）。
接下来，我们用分词器的掩码标记 ID 替换我们想要掩码的单词的字节 ID。论文作者指出，如果被掩码的片段以空格开头，模型的表现会好得多。

In [ ]:
text = "This is an incomplete sentence where some words are missing."
encoding = tokenizer(text, padding="max_length", return_tensors="ms")
# 掩码 " missing."
encoding.input_ids[0, 52:61] = tokenizer.mask_token_id
inputs, input_mask = encoding.input_ids, encoding.attention_mask

In [ ]:
print("Inputs:", tokenizer.decode(inputs.squeeze()))

接下来，我们可以让模型进行一次前向传播。输入的形状为 (批量大小, 序列长度) = (1, 2048)。

模型会输出形状为 (批量大小, 序列长度, 词表大小) 的对数几率（logits），在这种情况下，其形状为 (1, 2048, 262)。Perceiver 模型的词表大小是 262，其中 256 对应字节，另外 6 个对应 6 个预留的特殊标记。 


In [ ]:
outputs = model(inputs=inputs, attention_mask=input_mask)
logits = outputs.logits
masked_tokens_predictions = logits[0, 51:61].argmax(-1)
print(tokenizer.decode(masked_tokens_predictions))

### 使用 Perceiver 处理图像
Perceiver 在处理图像方面也表现出色。在这里，我们加载猫咪图片。

In [ ]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image

Perceiver 的研发人员发布了 3 种用于图像分类的 Perceiver 变体（它们仅在预处理环节有所不同）。

在这里，我们加载第一个变体——`PerceiverForImageClassificationLearned`，该变体将可学习的绝对位置嵌入添加到像素值中。

我们可以使用 `PerceiverFeatureExtractor` 为模型准备图像，它会对图像进行中心裁剪、调整大小，并将图像归一化至 224x224 的分辨率。 

In [ ]:
from mindnlp.transformers.models import PerceiverFeatureExtractor, PerceiverForImageClassificationLearned

del model
feature_extractor = PerceiverFeatureExtractor.from_pretrained("deepmind/vision-perceiver-learned")
model = PerceiverForImageClassificationLearned.from_pretrained("deepmind/vision-perceiver-learned")

在这里，我们为模型准备好图像，然后将其输入模型进行前向传播。 

In [ ]:
encoding = feature_extractor(image, return_tensors="ms")
inputs, input_mask = encoding.pixel_values, None

outputs = model(inputs, input_mask)
logits = outputs.logits

模型会输出形状为 (批量大小, 类别数量) 的对数几率（logits），在这种情况下是 (1, 1000) —— 因为该模型是在包含 1000 个可能类别的 ImageNet - 1k 数据集上进行训练的。 

In [ ]:
print("Predicted class:", model.config.id2label[logits.argmax(-1).item()])